<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/Zadanie_albumy_wszechczas%C3%B3w.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📀 Zadanie: najlepiej sprzedające się albumy wszechczasów

In [ ]:
# Komórka 1 – import bibliotek
import pandas as pd
import requests
from io import StringIO

print("Biblioteki zaimportowane.")

W tej komórce importujemy niezbędne biblioteki:
- **pandas** – do analizy danych,
- **requests** – do pobrania strony z nagłówkami,
- **StringIO** – do przekształcenia tekstu strony w obiekt plikopodobny.

In [ ]:
# Komórka 2 – pobranie danych ze strony z odpowiednim nagłówkiem (unikanie błędu 403)
url = 'https://www.officialcharts.com/chart-news/the-best-selling-albums-of-all-time-on-the-official-uk-chart__15551/'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

response = requests.get(url, headers=headers)
print(f"Kod odpowiedzi: {response.status_code}")

# Jeśli strona została pobrana poprawnie, wczytujemy tabele
if response.status_code == 200:
    tabele = pd.read_html(StringIO(response.text), header=0)
    print(f"Liczba znalezionych tabel: {len(tabele)}")
else:
    print("Nie udało się pobrać strony. Sprawdź połączenie lub użyj alternatywnej metody.")

Tutaj pobieramy zawartość strony internetowej. Dodajemy nagłówek `User-Agent`, aby serwer nie zablokował żądania.  
Funkcja `pd.read_html` wyciąga wszystkie tabele HTML – wynikiem jest lista.

In [ ]:
# Komórka 3 – wybór właściwej tabeli i pierwszy podgląd
# Na podstawie analizy strony interesująca nas tabela ma indeks 0 (zawiera kolumny POS, TITLE, ARTIST, YEAR, HIGH POSN)
df = tabele[0]
print("Pierwsze 5 wierszy tabeli z albumami:")
df.head()

Wybieramy pierwszą tabelę z listy (indeks 0). Za pomocą `head()` podglądamy kilka pierwszych wierszy, aby upewnić się, że dane są poprawne.

Krok 2: Przygotowanie danych (transformacje zgodne z zadaniem)

In [ ]:
# Komórka 4 – zamiana nagłówków na polskie odpowiedniki
df.columns = ['POZ', 'TYTUŁ', 'ARTYSTA', 'ROK', 'MAX POZ']
print("Nagłówki po zmianie na polskie:")
df.columns.tolist()

Zgodnie z poleceniem zmieniamy nazwy kolumn na:
- `POZ` (zamiast POS),
- `TYTUŁ` (zamiast TITLE),
- `ARTYSTA` (zamiast ARTIST),
- `ROK` (zamiast YEAR),
- `MAX POZ` (zamiast HIGH POSN).

In [ ]:
# Komórka 5 – liczba pojedynczych artystów
liczba_artystow = df['ARTYSTA'].nunique()
print(f"Liczba pojedynczych artystów/zespołów na liście: {liczba_artystow}")

Używamy metody `nunique()`, aby policzyć unikalnych wykonawców.

In [ ]:
# Komórka 6 – najczęściej pojawiające się zespoły
top_artysci = df['ARTYSTA'].value_counts().head(5)
print("Zespoły/artyści pojawiający się najczęściej na liście (top 5):")
top_artysci

`value_counts()` zlicza wystąpienia każdego artysty. Wyświetlamy 5 najczęstszych.

In [ ]:
# Komórka 7 – zmiana nagłówków na format tytułowy (każde słowo z dużej litery)
df.columns = df.columns.str.title()
print("Nagłówki po zmianie na format tytułowy:")
df.columns.tolist()

Za pomocą `str.title()` przekształcamy nazwy kolumn tak, by każde słowo zaczynało się wielką literą.  
Przykład: `MAX POZ` → `Max Poz`, `TYTUŁ` → `Tytuł` (niestety polskie znaki mogą być pominięte, ale tak działa to polecenie).

In [ ]:
# Komórka 8 – usunięcie kolumny 'Max Poz'
df = df.drop(columns=['Max Poz'])
print("Kolumny po usunięciu 'Max Poz':")
df.columns.tolist()

Pozbywamy się kolumny `Max Poz` zgodnie z treścią zadania.

In [ ]:
# Komórka 9 – rok z największą liczbą albumów
rok_najwiecej = df['Rok'].value_counts().idxmax()
ilosc_w_roku = df['Rok'].value_counts().max()
print(f"Najwięcej albumów ({ilosc_w_roku}) wydano w roku: {rok_najwiecej}.")

`value_counts()` na kolumnie `Rok` pokazuje, ile albumów pochodzi z danego roku. `idxmax()` zwraca rok z maksymalną liczbą, a `max()` – tę liczbę.

In [ ]:
# Komórka 10 – liczba albumów wydanych między 1960 a 1990 (włącznie)
albumy_1960_1990 = df[(df['Rok'] >= 1960) & (df['Rok'] <= 1990)]
print(f"Liczba albumów wydanych w latach 1960-1990: {len(albumy_1960_1990)}")

Filtrujemy wiersze, w których rok jest z żądanego zakresu, a następnie liczymy je funkcją `len()`.

In [ ]:
# Komórka 11 – najmłodszy album (najpóźniejszy rok)
najmlodszy_rok = df['Rok'].max()
print(f"Najmłodszy album (o najpóźniejszym roku wydania) pochodzi z: {najmlodszy_rok}.")

`max()` na kolumnie `Rok` daje największą wartość, czyli najpóźniejszy rok wydania.

In [ ]:
# Komórka 12 – lista najwcześniej wydanych albumów każdego artysty
# Dla każdego artysty znajdujemy wiersz z minimalnym rokiem
najwczesniejsze = df.loc[df.groupby('Artysta')['Rok'].idxmin()]
# Sortujemy alfabetycznie według artysty
najwczesniejsze = najwczesniejsze.sort_values('Artysta')
print("Najwcześniej wydane albumy każdego artysty (pierwsze 10 wierszy):")
najwczesniejsze.head(10)

Grupujemy dane po kolumnie `Artysta`, dla każdej grupy znajdujemy indeks wiersza z minimalnym rokiem (`idxmin()`), a następnie wybieramy te wiersze z oryginalnego DataFrame. Wynik sortujemy alfabetycznie.

Krok 3: Eksport wyników

In [ ]:
# Komórka 13 – zapis listy do pliku CSV
najwczesniejsze.to_csv('najwczesniejsze_albumy.csv', index=False)
print("Plik 'najwczesniejsze_albumy.csv' został zapisany w bieżącym katalogu.")

# Pobranie pliku na komputer (tylko w Colab)
try:
    from google.colab import files
    files.download('najwczesniejsze_albumy.csv')
    print("Plik został pobrany na Twój komputer.")
except ImportError:
    print("Jeśli pracujesz lokalnie, plik znajduje się w bieżącym katalogu.")

Zapisujemy DataFrame do pliku CSV bez indeksu. Jeśli pracujemy w Google Colab, dodatkowo pobieramy plik na lokalny komputer.